# TIE-ROLE-PILOT-001 — direct 2–3 hour compute GO/NO-GO

This is the **decision experiment** for the next expensive TIE-ROLE campaign. It directly compares the proposed balanced tied-gradient treatment against the canonical control on two fresh matched seeds.

At the end the notebook prints exactly one decision: **RUN_FULL_TIE_ROLE** or **DO_NOT_RUN_FULL_TIE_ROLE**. Weak or ambiguous evidence is automatically NO-GO.

It does not read or generate sealed rows, and it cannot modify the completed S5 verdicts.

**Colab:** select **Runtime → Change runtime type → T4 GPU**, then **Run all**. Google Drive is used so a disconnected session can resume from checkpoints. Keep the checkpoint ZIP in Drive even after the run finishes.

In [ ]:
import sys, json, hashlib, shutil, subprocess
import os
# Architecture: allocator must be fixed before ANY GPU process starts
# (env inheritance into the Cell-2 operator subprocess). Setting after
# torch import is too late. Never import torch in this setup kernel —
# query nvidia-smi only so the kernel holds no CUDA context while the
# operator subprocess needs the full single T4.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ['PYTHONUNBUFFERED'] = '1'
from pathlib import Path

REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
BRANCH = 'cymek-next-core-architecture'
EXECUTION_COMMIT = 'ffb52505f0d6a1029ef66a5d886abbf84794290c'
OPERATOR_BLOB = '084f29cc3fcdb2a7975ec654d590b825b30639f8'
PREREG_BLOB = '58df12ac54af1bd634b9423035b34597275f7bf7'
REPO = Path('/content/An-Ra-the-new-AGI-tie-role-pilot')
OP_COPY = Path('/content/tie_role_pilot_001_colab_v1.py')
PRE_COPY = Path('/content/TIE_ROLE_PILOT_001_PREREGISTRATION.json')

if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REMOTE,str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXECUTION_COMMIT],check=True)
head = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert head == EXECUTION_COMMIT, (head, EXECUTION_COMMIT)

def blob(path):
    return subprocess.check_output(['git','-C',str(REPO),'hash-object',path],text=True).strip()
assert blob('tools/tie_role_pilot_001_colab_v1.py') == OPERATOR_BLOB
assert blob('docs/cymek/experiments/TIE-ROLE-PILOT-001/PREREGISTRATION.json') == PREREG_BLOB
shutil.copy2(REPO/'tools/tie_role_pilot_001_colab_v1.py', OP_COPY)
shutil.copy2(REPO/'docs/cymek/experiments/TIE-ROLE-PILOT-001/PREREGISTRATION.json', PRE_COPY)


try:
    import tokenizers  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers'], check=True)

# Architecture: no torch import in this kernel (see top). nvidia-smi only.
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], capture_output=True, text=True)
gpus = [l.strip() for l in smi.stdout.strip().splitlines() if l.strip()] if smi.returncode == 0 else []
print('GPUs visible:', gpus)
if len(gpus) != 1 or 'T4' not in gpus[0]:
    raise RuntimeError('OFFICIAL PILOT BLOCKED: select Runtime → Change runtime type → T4 GPU (observed: ' + str(gpus) + ')')

from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/CYMEK/TIE_ROLE_PILOT_001')
OUT.mkdir(parents=True, exist_ok=True)
binding = {
  'schema': 'anra.tie-role-pilot-colab-binding/v1',
  'execution_commit': EXECUTION_COMMIT,
  'operator_blob': OPERATOR_BLOB,
  'prereg_blob': PREREG_BLOB
}
bp = OUT/'COLAB_EXECUTABLE_BINDING.json'
if bp.exists(): assert json.loads(bp.read_text()) == binding, 'Existing Drive state belongs to a different executable; use a new folder.'
else: bp.write_text(json.dumps(binding, indent=2)+'\n')
print('READY')
print('Execution commit:', EXECUTION_COMMIT)
print('Drive output:', OUT)

In [ ]:
cmd = [sys.executable, '-u', str(OP_COPY), '--repo', str(REPO), '--out', str(OUT), '--prereg', str(PRE_COPY), '--execution-commit', EXECUTION_COMMIT]
print('Starting/resuming TIE-ROLE-PILOT-001...', flush=True)
print('Target: ~2–3 hours on one T4.', flush=True)
proc = subprocess.run(cmd, cwd=REPO)
print('RETURN CODE:', proc.returncode)
if proc.returncode != 0:
    raise SystemExit('Pilot stopped/failed. DO NOT delete the Drive folder. Reconnect a T4 and Run all; compatible checkpoints exact-resume.')
result = json.loads((OUT/'TIE_ROLE_PILOT_RESULT.json').read_text())
print('\nDECISION:', result['decision'])
print('NEXT EXPENSIVE EXPERIMENT WORTH RUNNING:', result['expensive_experiment_worth_running'])
print('REASON:', result['reason'])

In [ ]:
from google.colab import files
result_path = OUT/'TIE_ROLE_PILOT_RESULT.json'
results_zip = OUT/'TIE_ROLE_PILOT_001_RESULTS.zip'
checkpoints_zip = OUT/'TIE_ROLE_PILOT_001_CHECKPOINTS.zip'
if not result_path.exists() or not results_zip.exists() or not checkpoints_zip.exists():
    raise RuntimeError('Pilot is not complete. Preserve Drive state and resume Cell 2.')
result = json.loads(result_path.read_text())
print('FINAL DECISION:', result['decision'])
print('EXPENSIVE EXPERIMENT WORTH RUNNING:', result['expensive_experiment_worth_running'])
print('Results SHA256:', hashlib.sha256(results_zip.read_bytes()).hexdigest())
print('CHECKPOINT ARCHIVE PRESERVED IN DRIVE:', checkpoints_zip)
print('Downloading RESULTS ZIP. Upload it back to ChatGPT.')
files.download(str(results_zip))